In [ ]:

%pip install transformers datasets seqeval torch accelerate --quiet

In [ ]:
import torch
from transformers import ElectraTokenizerFast, ElectraForTokenClassification, Trainer, TrainingArguments
from datasets import Dataset
import evaluate
import numpy as np
from sklearn.model_selection import train_test_split


In [ ]:

def load_conll_data(filepath):
    sentences, labels = [], []
    tokens, tags = [], []

    with open("data/resumes.conll", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append(tokens)
                    labels.append(tags)
                    tokens, tags = [], []
            else:
                splits = line.split()
                if len(splits) >= 2:
                    tokens.append(splits[0])
                    tags.append(splits[-1])

    if tokens:  # last sentence
        sentences.append(tokens)
        labels.append(tags)

    return sentences, labels

train_sentences, train_labels = load_conll_data("data/train.conll")
val_sentences, val_labels = load_conll_data("data/valid.conll")

print("Example sentence:", train_sentences[0])
print("Example labels:", train_labels[0])

In [ ]:

unique_labels = sorted(list(set(tag for doc in train_labels for tag in doc)))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

print("Labels:", label2id)

In [ ]:

model_name = "google/electra-small-discriminator"
tokenizer = ElectraTokenizerFast.from_pretrained(model_name)

def tokenize_and_align_labels(sentences, labels):
    tokenized_inputs = tokenizer(
        sentences,
        truncation=True,
        is_split_into_words=True,
        padding=True,
    )

    all_labels = []
    for i, label in enumerate(labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

train_encodings = tokenize_and_align_labels(train_sentences, train_labels)
val_encodings = tokenize_and_align_labels(val_sentences, val_labels)

In [ ]:

train_dataset = Dataset.from_dict(train_encodings)
val_dataset = Dataset.from_dict(val_encodings)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})
dataset

In [ ]:

model = ElectraForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

In [ ]:

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=-1)

    true_labels = [
        [id2label[l] for l in label if l != -100]
        for label in labels
    ]
    true_predictions = [
        [id2label[pred] for (pred, lab) in zip(prediction, label) if lab != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:

training_args = TrainingArguments(
    output_dir="./electra_ner_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)



In [ ]:
trainer.train()

In [ ]:

metrics = trainer.evaluate()
print(metrics)

In [ ]:

trainer.save_model("./electra_ner_model3")
tokenizer.save_pretrained("./electra_ner_model3")

In [ ]:
# Load your fine-tuned model
ner_model = pipeline("ner", model="./electra_ner_model3",tokenizer="./electra_ner_model3",aggregation_strategy="simple")

# Try it on a sample text
text = """

    Riya S Nair  
Machine Learning Engineer  
Email: riya.nair24@gmail.com | Phone: +91 9876543210  
LinkedIn: www.linkedin.com/in/riyanair | GitHub: github.com/riyanair24  

Location: Kochi, Kerala, India  

PROFILE  
Innovative Machine Learning Engineer with 3 years of experience in developing NLP and Computer Vision models. Passionate about data-driven problem solving and model optimization.

SKILLS  
Python, TensorFlow, PyTorch, Scikit-learn, Pandas, NumPy, NLP, Computer Vision, Flask, AWS, SQL, Docker, Git  

EXPERIENCE  
Data Scientist | Accenture | Bengaluru, India | Jan 2022 – Present  
- Built and deployed machine learning models for customer churn prediction with 90% accuracy.  
- Automated feature extraction pipeline using AWS Lambda and EC2.  
- Led a team of 3 interns to optimize model inference time by 40%.

Machine Learning Intern | TCS | Kochi, India | Jun 2021 – Dec 2021  
- Developed a text classification model using BERT achieving 85% F1 score.  
- Implemented Flask API for serving predictions.  

EDUCATION  
B.Tech in Computer Science and Engineering  
Amrita School of Engineering, Coimbatore (2017 – 2021)  
CGPA: 8.6 / 10  

LANGUAGES  
English, Malayalam, Hindi

"""
preds = ner_model(text)
seen = set()
unique_preds = []
for p in preds:
    key = (p['start'], p['end'], p['entity_group'])
    if key not in seen:
        unique_preds.append(p)
        seen.add(key)

for p in unique_preds:
    print(f"Entity: {p['entity_group']}, Word: {p['word']}, Score: {p['score']:.2f}")